# Tools and structured output

Goal: register `@tool` / `pydantic_toolset` and an `output_type` against a scripted model.

Trust: T2 callback plus the Pydantic extra. Network: none.


In [ ]:
from typing import Any

from pydantic import BaseModel

import finstack_ai


class Answer(BaseModel):
    answer: int


@finstack_ai.tool
def add(left: int, right: int) -> Answer:
    """Add two integers."""
    return Answer(answer=left + right)


tools = finstack_ai.pydantic_toolset(
    add,
    component="notebook.toolset.math",
    name="math",
)
print(add.input_schema["required"])
print(tools.tool_count)

In [ ]:
calls = 0


async def tool_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context, request
    global calls
    calls += 1
    if calls == 1:
        return {
            "text": "",
            "completion_id": "notebook-tool-1",
            "tool_calls": [{"name": "add", "arguments": {"left": 20, "right": 22}}],
        }
    return {"text": "42", "completion_id": "notebook-tool-2"}


agent = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        tool_model,
        component="notebook.model.tools",
        provider="notebook",
        model="notebook-model",
    ),
    [tools],
)
result = await agent.run("Add 20 and 22")
print(result.text)
assert result.text == "42"

In [ ]:
async def structured_model(
    context: finstack_ai.CallbackContext, request: dict[str, Any]
) -> dict[str, Any]:
    del context, request
    return {"json": {"answer": 42}, "completion_id": "notebook-structured-1"}


structured = await finstack_ai.Agent.from_python(
    finstack_ai.PythonModel(
        structured_model,
        component="notebook.model.structured",
        provider="notebook",
        model="notebook-model",
    ),
    output_type=Answer,
)
typed = await structured.run("answer")
print(typed.output)
assert typed.output == Answer(answer=42)